In [7]:
import torch
from utils import load_splits
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import Dataset

/home/default/projects/fake_news_classification/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
#global random seed
SEED = 42
torch.manual_seed(SEED)

In [9]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

True
NVIDIA GeForce RTX 3050 6GB Laptop GPU


device(type='cuda')

In [10]:
X_train, X_val, X_test, y_train, y_val, y_test = load_splits()

1
2
3


In [11]:

def init_model(model_name):
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        id2label={0: "True", 1:"Fake"},
        label2id={"True": 0, "Fake": 0}
    )
    return model


In [12]:
from datasets import Dataset
from transformers import AutoTokenizer

def tokenize_dataset(x, y, model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    dataset = Dataset.from_dict({
        "text": x.fillna("").astype(str).tolist(),
        "labels": y.astype(int).tolist()
    })

    def tokenize(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            max_length=256
        )

    return dataset.map(
        tokenize,
        batched=True
    )


In [13]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0,
    )

    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1,
    }

In [14]:
from transformers import (
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)

import torch
import gc
import json
from pathlib import Path


def training_loop(ds_train, ds_val, learning_rate, model_name, per_device_train_batch_size, gradient_accumulation_step):
    print(f"Training with learning rate: {learning_rate}")

    model = init_model(model_name)

    training_args = TrainingArguments(
        output_dir=f"models/{model_name}_{learning_rate}",
        learning_rate=learning_rate,
        per_device_train_batch_size=per_device_train_batch_size,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=gradient_accumulation_step,
        num_train_epochs=5,
        weight_decay=0.01,
        fp16=torch.cuda.is_available(),
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        save_total_limit=5,
        logging_strategy="epoch",
        report_to="none",
        seed=42,
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=ds_train,
        eval_dataset=ds_val,
        processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics,
    )

    trainer.train()

    results = []
    for log in log_history:
        if "loss" in log:
            epoch_results = log
        if "eval_loss" in log:
            epoch_results.update(log)
            results.append(epoch_results)

    # Create logs directory if necessary
    Path("logs").mkdir(parents=True, exist_ok=True)

    # Save a separate JSON file for each learning rate
    log_path = f"logs/{model_name}_lr_{learning_rate}.json"

    with open(log_path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=4)

    print(f"Saved results to {log_path}")

    del trainer, model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return results

In [ ]:

all_results = []
model = "distilbert-base-uncased"

for learning_rate in [1e-6, 5e-6, 1e-5, 2e-5, 3e-5]:
    ds_train = tokenize_dataset(X_train, y_train, model)
    ds_val =  tokenize_dataset(X_val, y_val, model)     
    all_results.extend(training_loop(ds_train, ds_val, learning_rate, model, 4, 4))



Map: 100%|██████████| 9333/9333 [00:00<00:00, 39249.21 examples/s]


Training with learning rate: 1e-06


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1293.55it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss


In [ ]:
# test
import json
from pathlib import Path

path = Path('models/bert-base-uncased_3e-05/checkpoint-13615/trainer_state.json')

with open(path, 'r') as f:
    trainer_state = json.load(f)


log_history = trainer_state["log_history"]

results = []
for log in log_history:
    if "loss" in log:
        epoch_results = log
    if "eval_f1_macro" in log:
        epoch_results.update(log)
        results.append(epoch_results)

results

[{'epoch': 1.0,
  'grad_norm': 0.05321201682090759,
  'learning_rate': 2.4002203452074918e-05,
  'loss': 1.7328090212426552,
  'step': 2723,
  'eval_accuracy': 0.9406407371691846,
  'eval_f1_macro': 0.9401980428424308,
  'eval_loss': 0.20408660173416138,
  'eval_precision_macro': 0.9388789330678448,
  'eval_recall_macro': 0.9427462141167475,
  'eval_runtime': 45.8818,
  'eval_samples_per_second': 203.414,
  'eval_steps_per_second': 50.87},
 {'epoch': 2.0,
  'grad_norm': 57.30626678466797,
  'learning_rate': 1.8002203452074918e-05,
  'loss': 1.2947496406479526,
  'step': 5446,
  'eval_accuracy': 0.9457837779920711,
  'eval_f1_macro': 0.9450101444536667,
  'eval_loss': 0.29040637612342834,
  'eval_precision_macro': 0.9458426613029889,
  'eval_recall_macro': 0.9442821331493751,
  'eval_runtime': 46.8636,
  'eval_samples_per_second': 199.153,
  'eval_steps_per_second': 49.804},
 {'epoch': 3.0,
  'grad_norm': 0.020094022154808044,
  'learning_rate': 1.2002203452074918e-05,
  'loss': 0.69339